# Job postings → ESCO

Maps LinkedIn job postings onto the ESCO classification: raw titles become ESCO occupations, raw skill terms become ESCO skills. Run the cells in order.

**Set the runtime to a GPU first** — *Runtime → Change runtime type → T4 GPU*. The cross-encoder rerank dominates the runtime and is roughly an order of magnitude slower on CPU.

Rough wall-clock on a T4, with the ESCO embeddings already cached:

| `SAMPLE_SIZE` | Time |
|---|---|
| 5,000 | a few minutes |
| 100,000 | ~1 hour |
| `None` (1.35M) | many hours — use a persistent cache (cell 5) |

Start small. Confirm the setup works end to end before committing to a long run.

## 1. Code and dependencies

In [ ]:
import os
import sys

# Point this at your fork, or skip the clone if you uploaded the folder
# yourself and just set REPO_ROOT to where it landed.
REPO_URL = "https://github.com/orihazan1/Linkedin_final_project.git"
REPO_ROOT = "/content/prep_data"

if not os.path.isdir(REPO_ROOT):
    !git clone $REPO_URL $REPO_ROOT

os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

!pip install -q -r requirements.txt

## 2. LinkedIn postings, via the Kaggle API

**This needs a free Kaggle account.** Register at [kaggle.com](https://www.kaggle.com/), then go to *Settings → API → Create New Token*.

The cell below takes credentials two ways, and prefers the first:

1. **Colab Secrets** — the key icon in the left sidebar. Add `KAGGLE_USERNAME` and `KAGGLE_KEY` (both are inside the `kaggle.json` you just downloaded), switch on *Notebook access*, and nothing is uploaded or written to disk. Secrets survive a runtime restart, so this is worth the one-time setup.
2. **Uploading `kaggle.json`** — the fallback when no secrets are set. Colab wipes local storage on restart, so this means re-uploading every session.

The token is a password equivalent: never commit it, and use *Expire Token* on that same Settings page if it leaks.

The download is about 1.1 GB. Only `linkedin_job_postings.csv` and `job_skills.csv` are read — `--unzip` also unpacks the ~5 GB `job_summary.csv`, which nothing uses.

In [ ]:
import os

DATA_DIR = "data/Linkedin Jobs & Skills (2024)"
NEEDED = ("linkedin_job_postings.csv", "job_skills.csv")


def have_data():
    return all(os.path.exists(os.path.join(DATA_DIR, f)) for f in NEEDED)


def have_credentials():
    """The Kaggle CLI reads the env vars first, then ~/.kaggle/kaggle.json."""
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    return os.path.exists("/root/.kaggle/kaggle.json")


if have_data():
    print("already downloaded - skipping")
else:
    # Preferred: Colab Secrets. Nothing touches disk, and it survives a restart.
    if not have_credentials():
        try:
            from google.colab import userdata

            user, key = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
            os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
            print("credentials: Colab Secrets")
        except Exception as e:
            print(f"no usable secrets ({type(e).__name__}), falling back to upload")

    # Fallback: pick your kaggle.json when the upload prompt appears.
    if not have_credentials():
        from google.colab import files

        files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        os.replace("kaggle.json", "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)  # the CLI rejects a readable token
        print("credentials: uploaded kaggle.json")

    # A 403 here means the dataset terms are unaccepted: open the dataset page on
    # kaggle.com while logged in, click through them, then rerun this cell.
    !kaggle datasets download -d asaniczka/1-3m-linkedin-jobs-and-skills-2024 -p "$DATA_DIR" --unzip

!ls -lh "$DATA_DIR"

## 3. ESCO classification

**This needs a free ESCO portal account, and is a manual download** — the ESCO dump is not on Kaggle and has no public API.

Register at [esco.ec.europa.eu/en/use-esco/download](https://esco.ec.europa.eu/en/use-esco/download), download **ESCO v1.2.1 · classification · English · CSV**, and upload the zip below. The folder must keep its published name, which the cell handles.

In [ ]:
from google.colab import files

ESCO_DIR = "data/ESCO dataset - v1.2.1 - classification - en - csv"

if not os.path.exists(os.path.join(ESCO_DIR, "occupations_en.csv")):
    uploaded = files.upload()          # the ESCO zip
    os.makedirs(ESCO_DIR, exist_ok=True)
    for name in uploaded:
        !unzip -o -j "$name" -d "$ESCO_DIR"

!ls "$ESCO_DIR"

## 4. Optional: keep the caches on Drive

Colab wipes local storage on restart. The ESCO embedding matrices take minutes to rebuild and a full run's output takes hours, so for anything beyond a quick test it is worth pointing the cache at Drive.

Skip this cell entirely for a short run — everything works without it.

`PREP_CACHE_DIR` must be set **before** importing anything from `src`, since the paths are resolved at import time.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
os.environ["PREP_CACHE_DIR"] = "/content/drive/MyDrive/prep_data_cache"
os.makedirs(os.environ["PREP_CACHE_DIR"], exist_ok=True)
print("cache ->", os.environ["PREP_CACHE_DIR"])

## 5. Check the device

Do this before starting a long run. On a CPU runtime the pipeline still works, but the cross-encoder stage becomes the difference between minutes and hours.

In [ ]:
import torch

from src import pipeline
from src.canonicalization import semantic_matching

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun from cell 1.")

print("pipeline will use:", semantic_matching.DEVICE)

## 6. Run the pipeline

`SAMPLE_SIZE=None` reads all 1.35M postings. Start with a few thousand.

Each sample size caches to its own file, so a quick test run cannot overwrite a long one.

In [ ]:
SAMPLE_SIZE = 5000

result = pipeline.run_pipeline(
    sample_size=SAMPLE_SIZE,
    use_cache=True,
    drop_unmatched=False,
    run_title_embeddings=True,
    run_skill_embeddings=True,
    skill_cross_encoder=True,
    save_output=True,
)

pipeline.print_pipeline_summary(result)

## 7. Inspect the matches

There is no ground-truth labelling to score against, so the thresholds are judged by reading samples. `print_calibration` shows the score and margin distributions with sampled pairs per band; the `_sample` and `_unmatched` functions show what each rung accepted and what it turned away.

In [ ]:
title_df, title_meta = result["title_df"], result["title_meta"]
skill_meta = result["skill_meta"]

semantic_matching.print_ce_sample(title_df)
semantic_matching.print_unmatched(title_df, title_meta["threshold"], n=20)
semantic_matching.print_skill_ce_sample(skill_meta)

In [ ]:
# Re-derive a threshold: score deciles, how many texts each cut-off would
# accept, and sample pairs per band to judge precision by eye.
semantic_matching.print_calibration(title_meta)

## 8. Export the CSV

In [ ]:
from src import export_job_skill_esco

out = export_job_skill_esco.build(sample_size=SAMPLE_SIZE, min_esco_skills=1)
out.to_csv(export_job_skill_esco.OUT_PATH, index=False, encoding="utf-8")
print(f"{len(out):,} rows -> {export_job_skill_esco.OUT_PATH}")

out.head()

In [ ]:
from google.colab import files

# Large files are slow to download through the browser; for a full run, copy it
# to Drive instead.
files.download(export_job_skill_esco.OUT_PATH)